In [46]:
import numpy as np
import pandas as pd
import pyarrow as pa
print(pd.__version__)
print(pa.__version__)

3.0.3
24.0.0


In [47]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "Name": ["Robert", "Alberto", "Piotr", "Marco"],
    "Country": ["UK", "Portugal", "Poland", "Italy"],
    "Cost": [550, 540, 200, 690],
})
df.dtypes

id         int64
Name         str
Country      str
Cost       int64
dtype: object

In [48]:
df.to_parquet("example.parquet")

# Reading Parquet files with pandas

## Serialization engines

Pandas supports two packages to read from Parquet files (serialize). One is fastparquet, the other pyarrow (the default).

In [49]:
pd.read_parquet("example.parquet", engine="pyarrow").dtypes

id         int64
Name         str
Country      str
Cost       int64
dtype: object

## Data type backend (experimental)

Pandas also gives the possibility to decide how the data in the created dataframe is stored:
- `numpy_nullable`: returns nullable-dtype-backed DataFrame
- `pyarrow`: returns pyarrow-backed nullable ArrowDtype DataFrame

With the default dtype backend, we will get the same data types / backends as we do when creating the dataframe from scratch, numpy int for integers and pyarrow string for string data (with pandas 3.0).

In [50]:
pd.read_parquet("example.parquet").dtypes

id         int64
Name         str
Country      str
Cost       int64
dtype: object

In [51]:
pd.read_parquet("example.parquet")._mgr.blocks

(NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64,
 ExtensionBlock: slice(2, 3, 1), 1 x 4, dtype: str,
 ExtensionBlock: slice(1, 2, 1), 1 x 4, dtype: str)

If we use the pyarrow dtype backend every column will be stored as a pyarrow chunked array.

**Note:** If the engine is `pyarrow` and dtype backend also `pyarrow` there are no conversion costs, only the serialization from the Parquet file!

In [52]:
pd.read_parquet("example.parquet", dtype_backend="pyarrow").dtypes

id                int64[pyarrow]
Name       large_string[pyarrow]
Country    large_string[pyarrow]
Cost              int64[pyarrow]
dtype: object

In [53]:
pd.read_parquet("example.parquet", dtype_backend="pyarrow")._mgr.blocks

(ExtensionBlock: slice(3, 4, 1), 1 x 4, dtype: int64[pyarrow],
 ExtensionBlock: slice(2, 3, 1), 1 x 4, dtype: large_string[pyarrow],
 ExtensionBlock: slice(1, 2, 1), 1 x 4, dtype: large_string[pyarrow],
 ExtensionBlock: slice(0, 1, 1), 1 x 4, dtype: int64[pyarrow])

In case of the ``numpy_nullable``option, pandas defines the mapping and selects int nullable backed by numpy for integers and pyarrow strings for string data.

In [54]:
pd.read_parquet("example.parquet", dtype_backend="numpy_nullable").dtypes

id          Int64
Name       string
Country    string
Cost        Int64
dtype: object

In [55]:
pd.read_parquet("example.parquet", dtype_backend="numpy_nullable")._mgr.blocks


(ExtensionBlock: slice(3, 4, 1), 1 x 4, dtype: Int64,
 ExtensionBlock: slice(2, 3, 1), 1 x 4, dtype: string,
 ExtensionBlock: slice(1, 2, 1), 1 x 4, dtype: string,
 ExtensionBlock: slice(0, 1, 1), 1 x 4, dtype: Int64)

In [56]:
pd.read_parquet("example.parquet", dtype_backend="numpy_nullable")._mgr.blocks[0].values

<IntegerArray>
[550, 540, 200, 690]
Length: 4, dtype: Int64

In [57]:
pd.read_parquet("example.parquet", dtype_backend="numpy_nullable")._mgr.blocks[0].values._mask

array([False, False, False, False])

In [58]:
pd.read_parquet("example.parquet", dtype_backend="numpy_nullable")._mgr.blocks[1].values

<ArrowStringArray>
['UK', 'Portugal', 'Poland', 'Italy']
Length: 4, dtype: string

Reading with fastparquet doesn't come with an option to define a data type backend

In [59]:
# import fastparquet
# pd.read_parquet("example.parquet", engine="fastparquet", dtype_backend="pyarrow")

# The above code will return a ValueError!

In [60]:
pd.read_parquet("example.parquet", engine="fastparquet").dtypes

id          int64
Name       object
Country    object
Cost        int64
dtype: object

In [61]:
pd.read_parquet("example.parquet", engine="fastparquet")._mgr.blocks

(NumpyBlock: slice(0, 6, 3), 2 x 4, dtype: int64,
 NumpyBlock: slice(1, 3, 1), 2 x 4, dtype: object)

When using a PyArrow engine, Pandas calls ``pq.read_table`` which returns a PyArrow Table (can be chunked) and then ``to_pandas`` to convert it to Pandas dataframe. When dtype backend is pyarrow, ``types_mapper`` argument is passed to ``to_pandas`` so that pyarrow creates blocks with PyArrow backed extension types.

In [62]:
import pyarrow.parquet as pq
t = pq.read_table('example.parquet')
t

pyarrow.Table
id: int64
Name: large_string
Country: large_string
Cost: int64
----
id: [[1,2,3,4]]
Name: [["Robert","Alberto","Piotr","Marco"]]
Country: [["UK","Portugal","Poland","Italy"]]
Cost: [[550,540,200,690]]

In [63]:
t.to_pandas().dtypes

id         int64
Name         str
Country      str
Cost       int64
dtype: object

In [64]:
t.to_pandas(types_mapper=pd.ArrowDtype).dtypes

id                int64[pyarrow]
Name       large_string[pyarrow]
Country    large_string[pyarrow]
Cost              int64[pyarrow]
dtype: object